1. **Preprocessing, normal augmentation, targeted augmentation in Section 1. (table)**
2. Ablation Study on Class Configuration in Section 4. (table)
3. Explainable AI on Feature - Feature Sufficiency and Necessity . (graph)
4. Ablation Study on BAN Components. (table)

# Section 0. Setup
- Experiment settings (constants)
- Setup with GPU
- Define the 2 model architectures at the beginning

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, recall_score
from sklearn.utils import class_weight
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import whisper
from transformers import pipeline
import torch
from collections import defaultdict
from datetime import datetime
import noisereduce as nr
from IPython.display import display

# Reproducibility for Reviewer Validation
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

class Config:
    # Data Settings
    DATA_PATH = 'dataset_meld/'
    FILE_PREFIX = 'prepared_'
    SR = 16000
    MAX_DURATION = 3
    MAX_SAMPLES = SR * MAX_DURATION
    
    # Feature Settings
    N_MFCC = 40
    MAX_MFCC_LEN = 94  
    
    # Training Parameters 
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 50
    DROPOUT = 0.3
    
    DEFAULT_CLASSES = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']

print("=== Hyperparameters ===")
for k, v in vars(Config).items():
    if not k.startswith('__'): print(f"{k}: {v}")

c:\Users\User\Downloads\[YTS]Github\FYP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Hyperparameters ===
DATA_PATH: dataset_meld/
FILE_PREFIX: prepared_
SR: 16000
MAX_DURATION: 3
MAX_SAMPLES: 48000
N_MFCC: 40
MAX_MFCC_LEN: 94
BATCH_SIZE: 32
LEARNING_RATE: 0.001
EPOCHS: 50
DROPOUT: 0.3
DEFAULT_CLASSES: ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']


In [2]:
import shutil
import imageio_ffmpeg

# 1. Get the directory where imageio_ffmpeg stores its binary
ffmpeg_bin_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())

# 2. Whisper looks specifically for "ffmpeg.exe". Let's make sure a copy exists with that exact name.
target_exe = os.path.join(ffmpeg_bin_dir, "ffmpeg.exe")
if not os.path.exists(target_exe):
    shutil.copy(imageio_ffmpeg.get_ffmpeg_exe(), target_exe)

# 3. Inject this directory into the system PATH for this session
os.environ["PATH"] += os.pathsep + ffmpeg_bin_dir

### Setup with GPU

In [3]:
import torch
print(torch.__version__)

2.11.0+cu128


In [4]:
# 1. Check GPU availability and set up device formatting
cuda_available = torch.cuda.is_available()
target_device = "cuda" if cuda_available else "cpu"
hf_device = 0 if cuda_available else -1

print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"Activating GPU: {torch.cuda.get_device_name(0)}\n")
else:
    print("GPU not detected. Running on CPU instead.\n")

CUDA Available: True
Activating GPU: NVIDIA GeForce RTX 5080 Laptop GPU



### Load Sentiment Classifer - RoBERTa

In [5]:
# Load RoBERTa Sentiment Classifier on the target device
print("Loading RoBERTa Sentiment Classifier...")
sentiment_classifier = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    device=hf_device
)

Loading RoBERTa Sentiment Classifier...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 67010.18it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Define Model Architectures

In [6]:
# ==========================================
# 1. HYBRID CNN-LSTM — BUILDER FUNCTION
# (Instantiation happens later, after data is loaded — see Section 2/3)
# ==========================================
def build_hybrid_cnn_lstm(num_classes, stats_dim, sent_dim):
    in_mfcc = layers.Input(shape=(Config.MAX_MFCC_LEN, Config.N_MFCC), name="in_mfcc")
    x_mfcc = layers.Conv1D(64, 3, activation='relu', padding='same')(in_mfcc)
    x_mfcc = layers.MaxPooling1D(2)(x_mfcc)
    x_mfcc = layers.LSTM(64)(x_mfcc)

    in_stats = layers.Input(shape=(stats_dim,), name="in_stats")
    x_stats = layers.Dense(32, activation='relu')(in_stats)
    x_stats = layers.BatchNormalization()(x_stats)

    in_text = layers.Input(shape=(sent_dim,), name="in_text")
    x_text = layers.Dense(16, activation='relu')(in_text)

    merged = layers.Concatenate()([x_mfcc, x_stats, x_text])
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(Config.DROPOUT)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=[in_mfcc, in_stats, in_text], outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=Config.LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [7]:
# ==========================================
# 2. BAN — BUILDER FUNCTION
# ==========================================
def build_ban(num_classes, stats_dim, sent_dim):
    in_mfcc = layers.Input(shape=(Config.MAX_MFCC_LEN, Config.N_MFCC), name="in_mfcc")
    x_mfcc = layers.Conv1D(64, 3, activation='relu', padding='same')(in_mfcc)
    x_mfcc = layers.MaxPooling1D(2)(x_mfcc)
    x_mfcc = layers.LSTM(64)(x_mfcc)

    in_stats = layers.Input(shape=(stats_dim,), name="in_stats")
    x_stats = layers.Dense(32, activation='relu')(in_stats)
    x_stats = layers.BatchNormalization()(x_stats)

    x_audio_combined = layers.Concatenate()([x_mfcc, x_stats])
    x_audio_proj = layers.Dense(64, activation='relu')(x_audio_combined)

    in_text = layers.Input(shape=(sent_dim,), name="in_text")
    x_text = layers.Dense(64, activation='relu')(in_text)

    attention_weights = layers.Dense(64, activation='sigmoid', name="text_guided_attention")(x_text)
    attended_audio = layers.Multiply(name="attended_audio")([x_audio_proj, attention_weights])

    merged = layers.Concatenate()([attended_audio, x_text])
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(Config.DROPOUT)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=[in_mfcc, in_stats, in_text], outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=Config.LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Section 1. Pre-processing and Data Augmentation
- Silence Removal x Noise Reduction
- Normal Data Augmentation vs Targeted Data Augmentation

**Two-stage design:**
- **Stage A**: Compare preprocessing options alone (no augmentation) → select the best.
- **Stage B**: Apply the winning preprocessing, then compare augmentation strategies on top of it

In [8]:
# Preprocessing function for audio waveform
def preprocess_waveform(y, sr, clean_noise=False, trim_silence=False):
    """
    Preprocessing so its effect stays isolated from augmentation
    """
    if clean_noise:
        try:
            if len(y) > sr * 0.1:
                y = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.8)
        except Exception:
            pass
    if trim_silence:
        try:
            y, _ = librosa.effects.trim(y, top_db=20)
        except Exception:
            pass
    return y

# Settings for TARGETED AUGMENTATION
STRATEGY = {
    'neutral': [],
    'joy':     ['time_stretch_fast', 'pitch_shift_up'],
    'sadness': ['time_stretch_slow', 'pitch_shift_down'],
    'anger':   ['noise', 'pitch_shift_up', 'time_stretch_fast'],
    'fear':    ['pitch_shift_up', 'time_stretch_fast', 'noise'],
    'disgust': ['time_stretch_slow', 'pitch_shift_down', 'noise'],
    'surprise':['pitch_shift_up', 'time_stretch_fast', 'noise'] 
}

def apply_augmentation(y, sr, aug_type):
    """aug_type='random' = generic normal augmentation; others = targeted methods."""
    try:
        if aug_type == 'random':
            choice = np.random.randint(0, 3)
            if choice == 0:
                return librosa.effects.time_stretch(y, rate=np.random.uniform(0.8, 1.2))
            elif choice == 1:
                return librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.randint(-2, 3))
            else:
                noise_amp = 0.005 * np.random.uniform() * np.amax(y)
                return y + noise_amp * np.random.normal(size=y.shape[0])
        elif aug_type == 'pitch_shift_up':
            return librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.uniform(1.5, 2.5))
        elif aug_type == 'pitch_shift_down':
            return librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.uniform(-2.5, -1.5))
        elif aug_type == 'time_stretch_fast':
            return librosa.effects.time_stretch(y, rate=np.random.uniform(1.1, 1.25))
        elif aug_type == 'time_stretch_slow':
            return librosa.effects.time_stretch(y, rate=np.random.uniform(0.75, 0.9))
        elif aug_type == 'noise':
            noise_amp = 0.005 * np.random.uniform() * np.amax(y)
            return y + noise_amp * np.random.normal(size=y.shape[0])
        return y
    except Exception:
        return y

In [9]:
# LOAD RAW SPLITS + PRECOMPUTE SENTIMENT (once — reused by every config)
def build_sentiment_annotated_df(split):
    path = os.path.join(Config.DATA_PATH, f"{split}_sent_emo.csv")
    df = pd.read_csv(path)
    df = df[df['Emotion'].str.lower().isin(Config.DEFAULT_CLASSES)].copy()

    df['file_path'] = df.apply(
        lambda r: os.path.join(Config.DATA_PATH, "audio", f"wav_{split}",
                                f"dia{r['Dialogue_ID']}_utt{r['Utterance_ID']}.wav"),
        axis=1
    )

    preds = sentiment_classifier(df['Utterance'].astype(str).tolist())
    df['textual_sentiment_predicted'] = [p['label'].lower() for p in preds]
    return df

print("Pre-annotating splits with RoBERTa sentiment (once, reused across all configs)...")
SPLIT_DFS = {split: build_sentiment_annotated_df(split) for split in ['train', 'dev', 'test']}
for split, df in SPLIT_DFS.items():
    print(f"  {split}: {len(df)} samples")

Pre-annotating splits with RoBERTa sentiment (once, reused across all configs)...
  train: 9989 samples
  dev: 1109 samples
  test: 2610 samples


In [10]:
# ==========================================
# UNIFIED FEATURE EXTRACTOR (routes 'none' / 'normal' / 'targeted')
# ==========================================
def process_single_row(row, clean_noise, trim_silence, aug_type=None):
    # --- Guard 1: file existence check (fast path, avoids librosa exception overhead) ---
    if not os.path.exists(row['file_path']):
        return None, 'missing_file'

    # --- Guard 2: load failure (corrupt/unreadable audio) ---
    try:
        y, _ = librosa.load(row['file_path'], sr=Config.SR)
    except Exception:
        return None, 'load_failed'

    if len(y) < 500:
        return None, 'too_short'

    y = preprocess_waveform(y, Config.SR, clean_noise, trim_silence)
    if aug_type is not None:
        y = apply_augmentation(y, Config.SR, aug_type)
    if len(y) < 500:
        return None, 'too_short_after_processing'

    if len(y) > Config.MAX_SAMPLES:
        y = y[:Config.MAX_SAMPLES]
    else:
        y = np.pad(y, (0, int(Config.MAX_SAMPLES - len(y))), 'constant')

    mfcc = librosa.feature.mfcc(y=y, sr=Config.SR, n_mfcc=Config.N_MFCC).T
    if mfcc.shape[0] > Config.MAX_MFCC_LEN:
        mfcc = mfcc[:Config.MAX_MFCC_LEN, :]
    else:
        mfcc = np.pad(mfcc, ((0, Config.MAX_MFCC_LEN - mfcc.shape[0]), (0, 0)), 'constant')

    try:
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=Config.SR).T, axis=0)
        contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=Config.SR).T, axis=0)
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=y).T, axis=0)
        rms = np.mean(librosa.feature.rms(y=y).T, axis=0)
        stats = np.hstack([chroma, contrast, zcr, rms])
    except Exception:
        return None, 'stats_extraction_failed'

    return (mfcc, stats), None


def extract_features_for_config(df, clean_noise=False, trim_silence=False,
                                 augmentation='none', is_train=True, target_count=1700):
    X_mfcc, X_stats, X_sent, y_labels = [], [], [], []
    skip_counts = defaultdict(int)

    if augmentation == 'targeted' and is_train:
        for emotion in Config.DEFAULT_CLASSES:
            sub_df = df[df['Emotion'].str.lower() == emotion]
            if len(sub_df) == 0:
                continue
            if len(sub_df) > target_count:
                sub_df = sub_df.sample(n=target_count, random_state=SEED)
                multiplier = 1
            else:
                multiplier = max(1, int(np.ceil(target_count / len(sub_df))))
            aug_methods = STRATEGY.get(emotion, ['random'])

            for _, row in tqdm(sub_df.iterrows(), total=len(sub_df), desc=f"[{emotion}]"):
                for i in range(multiplier):
                    aug_type = aug_methods[i % len(aug_methods)] if (i > 0 and aug_methods) else None
                    result, fail_reason = process_single_row(row, clean_noise, trim_silence, aug_type)
                    if result is None:
                        skip_counts[fail_reason] += 1
                        continue
                    mfcc, stats = result
                    X_mfcc.append(mfcc); X_stats.append(stats)
                    X_sent.append(row['textual_sentiment_predicted'])
                    y_labels.append(emotion)
    else:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            aug_type = 'random' if (augmentation == 'normal' and is_train) else None
            result, fail_reason = process_single_row(row, clean_noise, trim_silence, aug_type)
            if result is None:
                skip_counts[fail_reason] += 1
                continue
            mfcc, stats = result
            X_mfcc.append(mfcc); X_stats.append(stats)
            X_sent.append(row['textual_sentiment_predicted'])
            y_labels.append(row['Emotion'].lower())

    total_skipped = sum(skip_counts.values())
    print(f"  ⚠️ Skipped {total_skipped} samples: {dict(skip_counts)}")

    return np.array(X_mfcc), np.array(X_stats), np.array(X_sent), np.array(y_labels), dict(skip_counts)

In [11]:
# DATA PREP FOR A GIVEN CONFIG + TRAINING WRAPPER
def prepare_and_encode(clean_noise, trim_silence, augmentation):
    label_encoder = LabelEncoder().fit(Config.DEFAULT_CLASSES)
    sent_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    splits_out = {}
    skip_report = {}
    for split in ['train', 'dev', 'test']:
        m, s, txt, y, skips = extract_features_for_config(
            SPLIT_DFS[split], clean_noise=clean_noise, trim_silence=trim_silence,
            augmentation=augmentation, is_train=(split == 'train')
        )
        splits_out[split] = {'mfcc': m, 'stats': s, 'sent_raw': txt, 'y_raw': y}
        skip_report[split] = skips

    scaler = StandardScaler().fit(splits_out['train']['stats'])
    sent_encoder.fit(splits_out['train']['sent_raw'].reshape(-1, 1))

    for split in splits_out:
        splits_out[split]['stats'] = scaler.transform(splits_out[split]['stats'])
        splits_out[split]['sent'] = sent_encoder.transform(splits_out[split]['sent_raw'].reshape(-1, 1))
        splits_out[split]['y'] = label_encoder.transform(splits_out[split]['y_raw'])

    return splits_out, skip_report


def run_training(model_builder, splits_out, model_name=""):
    tf.keras.backend.clear_session(); gc.collect()

    X_train = [splits_out['train']['mfcc'], splits_out['train']['stats'], splits_out['train']['sent']]
    X_dev   = [splits_out['dev']['mfcc'],   splits_out['dev']['stats'],   splits_out['dev']['sent']]
    X_test  = [splits_out['test']['mfcc'],  splits_out['test']['stats'],  splits_out['test']['sent']]
    y_train, y_dev, y_test = splits_out['train']['y'], splits_out['dev']['y'], splits_out['test']['y']

    model = model_builder(len(Config.DEFAULT_CLASSES), X_train[1].shape[1], X_train[2].shape[1])
    cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

    hist = model.fit(
        X_train, y_train, validation_data=(X_dev, y_dev),
        epochs=Config.EPOCHS, batch_size=Config.BATCH_SIZE, verbose=0,
        class_weight=dict(enumerate(cw)),
        callbacks=[callbacks.EarlyStopping(patience=6, restore_best_weights=True)]
    )

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    return {
        'model_name': model_name,
        'val_accuracy': max(hist.history['val_accuracy']),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_macro_recall': recall_score(y_test, y_pred, average='macro'),
        'test_macro_f1': f1_score(y_test, y_pred, average='macro'),
    }

In [12]:
# ==========================================
# STAGE A: PREPROCESSING ONLY (no augmentation)
# ==========================================
PREPROCESS_CONFIGS = {
    "P0_None":        {"clean_noise": False, "trim_silence": False},
    "P1_SilenceOnly":  {"clean_noise": False, "trim_silence": True},
    "P2_NoiseOnly":    {"clean_noise": True,  "trim_silence": False},
    "P3_Both":         {"clean_noise": True,  "trim_silence": True},
}

stageA_hybrid, stageA_ban, stageA_skips = {}, {}, {}
stageA_splits_cache = {}

for name, cfg in PREPROCESS_CONFIGS.items():
    print(f"\n=== Stage A: {name} ===")
    splits_out, skip_report = prepare_and_encode(cfg['clean_noise'], cfg['trim_silence'], augmentation='none')
    stageA_splits_cache[name] = (splits_out, cfg)
    stageA_skips[name] = skip_report

    stageA_hybrid[name] = run_training(build_hybrid_cnn_lstm, splits_out, "Hybrid")
    stageA_ban[name]    = run_training(build_ban, splits_out, "BAN")

    total_skipped = sum(sum(v.values()) for v in skip_report.values())
    print(f"{name}: Hybrid test={stageA_hybrid[name]['test_accuracy']:.4f} | "
          f"BAN test={stageA_ban[name]['test_accuracy']:.4f} | Total skipped samples: {total_skipped}")
    del splits_out; gc.collect()


=== Stage A: P0_None ===


100%|██████████| 9989/9989 [02:07<00:00, 78.41it/s] 


  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 1109/1109 [00:14<00:00, 76.65it/s]


  ⚠️ Skipped 1 samples: {'missing_file': 1}


...

  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 1109/1109 [00:41<00:00, 26.77it/s]


  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 2610/2610 [01:41<00:00, 25.76it/s]


  ⚠️ Skipped 0 samples: {}
P3_Both: Hybrid test=0.3835 | BAN test=0.3670 | Total skipped samples: 2


In [13]:
df_stageA_hybrid = pd.DataFrame([{'Config': k, **v} for k, v in stageA_hybrid.items()]).drop(columns='model_name')
df_stageA_ban    = pd.DataFrame([{'Config': k, **v} for k, v in stageA_ban.items()]).drop(columns='model_name')

print("=== Table A1: Hybrid CNN-LSTM — Preprocessing ===")
display(df_stageA_hybrid)
print("\n=== Table A2: BAN — Preprocessing ===")
display(df_stageA_ban)

combined_A = df_stageA_hybrid.merge(df_stageA_ban, on='Config', suffixes=('_Hybrid', '_BAN'))
combined_A['Avg_Test_Accuracy']  = combined_A[['test_accuracy_Hybrid', 'test_accuracy_BAN']].mean(axis=1)
combined_A['Avg_Macro_Recall']   = combined_A[['test_macro_recall_Hybrid', 'test_macro_recall_BAN']].mean(axis=1)
combined_A['Avg_Macro_F1']       = combined_A[['test_macro_f1_Hybrid', 'test_macro_f1_BAN']].mean(axis=1)
combined_A = combined_A.sort_values('Avg_Macro_F1', ascending=False).reset_index(drop=True)

print("\n=== Combined Decision Table — Best Preprocessing (ranked by Macro F1) ===")
display(combined_A)

best_preprocess_name = combined_A.iloc[0]['Config']
best_preprocess_cfg = PREPROCESS_CONFIGS[best_preprocess_name]
print(f"\n✅ Best preprocessing: {best_preprocess_name} -> {best_preprocess_cfg}")

=== Table A1: Hybrid CNN-LSTM — Preprocessing ===


,Config,val_accuracy,test_accuracy,test_macro_recall,test_macro_f1
0,P0_None,0.358303,0.403831,0.292970,0.247886
1,P1_SilenceOnly,0.370939,0.409195,0.305168,0.257214
2,P2_NoiseOnly,0.380866,0.395402,0.297027,0.252442
3,P3_Both,0.370939,0.383525,0.276683,0.240777



=== Table A2: BAN — Preprocessing ===


,Config,val_accuracy,test_accuracy,test_macro_recall,test_macro_f1
0,P0_None,0.394404,0.370498,0.275895,0.236991
1,P1_SilenceOnly,0.380866,0.407663,0.289310,0.241416
2,P2_NoiseOnly,0.407942,0.437931,0.296468,0.235677
3,P3_Both,0.400722,0.367050,0.283973,0.229751



=== Combined Decision Table — Best Preprocessing (ranked by Macro F1) ===


,Config,val_accuracy_Hybrid,test_accuracy_Hybrid,test_macro_recall_Hybrid,test_macro_f1_Hybrid,val_accuracy_BAN,test_accuracy_BAN,test_macro_recall_BAN,test_macro_f1_BAN,Avg_Test_Accuracy,Avg_Macro_Recall,Avg_Macro_F1
0,P1_SilenceOnly,0.370939,0.409195,0.305168,0.257214,0.380866,0.407663,0.289310,0.241416,0.408429,0.297239,0.249315
1,P2_NoiseOnly,0.380866,0.395402,0.297027,0.252442,0.407942,0.437931,0.296468,0.235677,0.416667,0.296748,0.244060
2,P0_None,0.358303,0.403831,0.292970,0.247886,0.394404,0.370498,0.275895,0.236991,0.387165,0.284433,0.242439
3,P3_Both,0.370939,0.383525,0.276683,0.240777,0.400722,0.367050,0.283973,0.229751,0.375287,0.280328,0.235264



✅ Best preprocessing: P1_SilenceOnly -> {'clean_noise': False, 'trim_silence': True}


In [14]:
# ==========================================
# STAGE B: AUGMENTATION, using the winning preprocessing from Stage A
# ==========================================
AUGMENTATION_MODES = {"A1_NormalAug": "normal", "A2_TargetedAug": "targeted"}

stageB_hybrid, stageB_ban, stageB_skips = {}, {}, {}
stageB_splits_cache = {}

# A0 (no augmentation) = the winning Stage-A run itself — reuse instead of recomputing
stageB_hybrid["A0_NoAug"] = stageA_hybrid[best_preprocess_name]
stageB_ban["A0_NoAug"]    = stageA_ban[best_preprocess_name]

for name, aug_mode in AUGMENTATION_MODES.items():
    print(f"\n=== Stage B: {name} (on {best_preprocess_name}) ===")
    splits_out, skip_report = prepare_and_encode(cfg['clean_noise'], cfg['trim_silence'], augmentation='none')
    stageB_splits_cache[name] = (splits_out, cfg)
    stageB_skips[name] = skip_report

    stageB_hybrid[name] = run_training(build_hybrid_cnn_lstm, splits_out, "Hybrid")
    stageB_ban[name]    = run_training(build_ban, splits_out, "BAN")

    print(f"{name}: Hybrid test={stageB_hybrid[name]['test_accuracy']:.4f} | "
          f"BAN test={stageB_ban[name]['test_accuracy']:.4f}")
    del splits_out; gc.collect()


=== Stage B: A1_NormalAug (on P1_SilenceOnly) ===


100%|██████████| 9989/9989 [06:16<00:00, 26.54it/s]


  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 1109/1109 [00:41<00:00, 26.76it/s]


  ⚠️ Skipped 1 samples: {'missing_file': 1}


...

  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 1109/1109 [00:41<00:00, 26.80it/s]


  ⚠️ Skipped 1 samples: {'missing_file': 1}


100%|██████████| 2610/2610 [01:41<00:00, 25.70it/s]


  ⚠️ Skipped 0 samples: {}
A2_TargetedAug: Hybrid test=0.3701 | BAN test=0.3812


In [15]:
df_stageB_hybrid = pd.DataFrame([{'Config': k, **v} for k, v in stageB_hybrid.items()]).drop(columns='model_name')
df_stageB_ban    = pd.DataFrame([{'Config': k, **v} for k, v in stageB_ban.items()]).drop(columns='model_name')

print("=== Table B1: Hybrid CNN-LSTM — Augmentation ===")
display(df_stageB_hybrid)
print("\n=== Table B2: BAN — Augmentation ===")
display(df_stageB_ban)

combined_B = df_stageB_hybrid.merge(df_stageB_ban, on='Config', suffixes=('_Hybrid', '_BAN'))
combined_B['Avg_Test_Accuracy'] = combined_B[['test_accuracy_Hybrid', 'test_accuracy_BAN']].mean(axis=1)
combined_B = combined_B.sort_values('Avg_Test_Accuracy', ascending=False).reset_index(drop=True)

print("\n=== Combined Decision Table — Best Augmentation ===")
display(combined_B)

best_aug_name = combined_B.iloc[0]['Config']
print(f"\n✅ FINAL CONFIG for the rest of the notebook:")
print(f"   Preprocessing: {best_preprocess_name} -> {best_preprocess_cfg}")
print(f"   Augmentation:  {best_aug_name}")

=== Table B1: Hybrid CNN-LSTM — Augmentation ===


,Config,val_accuracy,test_accuracy,test_macro_recall,test_macro_f1
0,A0_NoAug,0.370939,0.409195,0.305168,0.257214
1,A1_NormalAug,0.367329,0.394253,0.278344,0.238253
2,A2_TargetedAug,0.340253,0.370115,0.274084,0.238585



=== Table B2: BAN — Augmentation ===


,Config,val_accuracy,test_accuracy,test_macro_recall,test_macro_f1
0,A0_NoAug,0.380866,0.407663,0.289310,0.241416
1,A1_NormalAug,0.361011,0.380843,0.284490,0.238567
2,A2_TargetedAug,0.375451,0.381226,0.265445,0.212302



=== Combined Decision Table — Best Augmentation ===


,Config,val_accuracy_Hybrid,test_accuracy_Hybrid,test_macro_recall_Hybrid,test_macro_f1_Hybrid,val_accuracy_BAN,test_accuracy_BAN,test_macro_recall_BAN,test_macro_f1_BAN,Avg_Test_Accuracy
0,A0_NoAug,0.370939,0.409195,0.305168,0.257214,0.380866,0.407663,0.289310,0.241416,0.408429
1,A1_NormalAug,0.367329,0.394253,0.278344,0.238253,0.361011,0.380843,0.284490,0.238567,0.387548
2,A2_TargetedAug,0.340253,0.370115,0.274084,0.238585,0.375451,0.381226,0.265445,0.212302,0.375670



✅ FINAL CONFIG for the rest of the notebook:
   Preprocessing: P1_SilenceOnly -> {'clean_noise': False, 'trim_silence': True}
   Augmentation:  A0_NoAug


In [16]:
# ==========================================
# MISSING-SAMPLE SUMMARY (for documentation)
# ==========================================
rows = []
for name, skip_report in {**stageA_skips, **stageB_skips}.items():
    for split, reasons in skip_report.items():
        for reason, count in reasons.items():
            rows.append({'Config': name, 'Split': split, 'Reason': reason, 'Count': count})

df_skips = pd.DataFrame(rows)
print("=== Missing / Skipped Sample Report (all configs) ===")
display(df_skips.pivot_table(index='Config', columns='Reason', values='Count', aggfunc='sum', fill_value=0))

print(f"\nTotal skipped across all configs and splits: {df_skips['Count'].sum()}")

=== Missing / Skipped Sample Report (all configs) ===


Reason,missing_file
Config,
A1_NormalAug,2
A2_TargetedAug,2
P0_None,2
P1_SilenceOnly,2
P2_NoiseOnly,2
P3_Both,2



Total skipped across all configs and splits: 12


### Best Preprocessing Configuration
+ Silence Removal

# Section 2. Feature Extraction

### 2.1 Textual Sentiment
Exclude ASR, import RoBERTa, classify textual sentiment from transcription

In [ ]:
AUDIO_DIRECTORY = "dataset_meld/audio/"
AUDIO_EXT = ".wav"
splits = ['train', 'dev', 'test']

os.makedirs("dataset_meld", exist_ok=True)

# Create a timestamp string for the filenames (e.g., "20260617_2030")
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# --- GLOBAL STATS (per split) ---
stats = defaultdict(int)
failed_samples = []

# Lists to collect data for final graphs
global_sentiment_scores = []

# --- PROCESSING LOOP ---
for split in splits:
    print(f"\n================ {split.upper()} SPLIT ================\n")

    orig_csv = os.path.join("dataset_meld", f"{split}_sent_emo.csv")
    out_csv = os.path.join("dataset_meld", f"prepared_{split}_{timestamp}_sent_emo.csv")

    print(f"Processing: {orig_csv}")
    df = pd.read_csv(orig_csv)

    # --- MOCK RUN MECHANISM ---
    MOCK_MODE = False  # Toggle this to False to run the full dataset
    if MOCK_MODE:
        # Take 10% of the data, sampled randomly
        df = df.sample(frac=0.10, random_state=SEED) 
        print(f"⚠️ MOCK MODE ACTIVE: Processing only 10% of {split} ({len(df)} samples)")
    # --------------------------

    prepared_data = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']

        raw_path = os.path.join(
            AUDIO_DIRECTORY,
            f"wav_{split}",
            f"dia{dia_id}_utt{utt_id}{AUDIO_EXT}"
        )
        file_path = os.path.abspath(raw_path)

        # ---------------------------
        # Missing audio
        # ---------------------------
        if not os.path.exists(file_path):
            stats["missing_audio"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "missing_audio"
            })
            continue

        # ---------------------------
        # Validate CSV fields
        # ---------------------------
        if pd.isna(row['Emotion']) or pd.isna(row['Sentiment']) or pd.isna(row['Utterance']):
            stats["invalid_csv_data"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "invalid_csv_data"
            })
            continue

        emo_gt = str(row['Emotion']).lower()
        sent_gt = str(row['Sentiment']).lower()
        trans_gt = str(row['Utterance']).strip()

        # ---------------------------
        # Sentiment prediction (on ground-truth transcription)
        # ---------------------------
        try:
            bert_out = sentiment_classifier(trans_gt)[0]
            sent_pred = bert_out['label'].lower()
            sent_score = bert_out['score']

        except Exception as e:
            stats["sentiment_failed"] += 1
            failed_samples.append({
                "split": split, "file_path": file_path, "reason": "sentiment_failed", "error": str(e)
            })
            continue

        # ---------------------------
        # SUCCESS CASE
        # ---------------------------
        prepared_data.append({
            'emotion': emo_gt,
            'sentiment': sent_gt,  
            'textual_sentiment_predicted': sent_pred,
            'textual_sentiment_score': sent_score,
            'transcription_ground_truth': trans_gt,
            'file_path': file_path
        })

        stats["success"] += 1
        global_sentiment_scores.append(sent_score)

    # ---------------------------
    # Save entire batch at once (Optimised for GPU speed)
    # ---------------------------
    if prepared_data:
        chunk_df = pd.DataFrame(prepared_data)
        chunk_df.to_csv(out_csv, index=False)

    print(f"\n✅ Finished {split}. Saved to {out_csv}")
# ---------------------------
# FINAL REPORT
# ---------------------------
print("\n================ FINAL DATASET REPORT ================\n")

total_failures = (
    stats["missing_audio"] + stats["invalid_csv_data"] +
    stats["sentiment_failed"]
)

# Calculate Averages
avg_sent = sum(global_sentiment_scores) / len(global_sentiment_scores) if global_sentiment_scores else 0

print(f"Success samples           : {stats['success']}")
print(f"Missing audio             : {stats['missing_audio']}")
print(f"Invalid CSV data          : {stats['invalid_csv_data']}")
print(f"Sentiment prediction fail : {stats['sentiment_failed']}")
print(f"TOTAL FAILURES            : {total_failures}")
print("-" * 54)
print(f"Average Sentiment Score   : {avg_sent:.4f}")

# ---------------------------
# SAVE FAILURE LOG
# ---------------------------
failed_df = pd.DataFrame(failed_samples)
log_filename = f"dataset_meld/failed_samples_log_{timestamp}.csv"
failed_df.to_csv(log_filename, index=False)

print(f"\nFailure log saved: {log_filename}")

# ---------------------------
# GENERATE DISTRIBUTION GRAPH
# ---------------------------
if global_sentiment_scores:
    plt.figure(figsize=(7, 5))

    sns.histplot(global_sentiment_scores, bins=50, kde=True, color='salmon')
    plt.title("Distribution of RoBERTa Sentiment Confidence")
    plt.xlabel("Confidence Score (0.0 to 1.0)")
    plt.ylabel("Number of Samples")
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

### 2.2 Acoustic Feature & 2.3 Statistical Features
MFCCs <br>
Chroma, contrast, zcr, rms

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
SR = 16000
MAX_DURATION = 3
MAX_SAMPLES = SR * MAX_DURATION
N_MFCC = 40
MAX_MFCC_LEN = 94 

previous_timestamp = "20260713_2030"

csv_files = {
    'train': os.path.join(Config.DATA_PATH, f"prepared_train_{previous_timestamp}_sent_emo.csv"),
    'dev': os.path.join(Config.DATA_PATH, f"prepared_dev_{previous_timestamp}_sent_emo.csv"),
    'test': os.path.join(Config.DATA_PATH, f"prepared_test_{previous_timestamp}_sent_emo.csv")
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
path = os.path.join(Config.DATA_PATH, f"all_features_{split}_{timestamp}.npz")

print(f"Starting Independent Feature Extraction to NPZ (Timestamp: {timestamp})...\n")

# ==========================================
# 2. EXTRACTION & SAVING LOOP (Per Split)
# ==========================================
for split, path in csv_files.items():
    if not os.path.exists(path):
        print(f"File not found: {path}")
        continue
        
    print(f"Processing {split.upper()} set from {path}")
    df = pd.read_csv(path)
    
    successful_rows = []
    mfcc_data = []
    stats_data = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        file_path = row['file_path']
        
        try:
            # 1. Load Audio
            y, _ = librosa.load(file_path, sr=SR)
            
            # 2. Safeguard against ultra-short clips
            if len(y) < 500:
                continue
                
            # 3. Pad or Crop to exactly 3 seconds
            if len(y) > MAX_SAMPLES:
                y = y[:MAX_SAMPLES]
            else:
                y = np.pad(y, (0, int(MAX_SAMPLES - len(y))), 'constant')
                
            # 4. Extract MFCC
            mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC).T
            if mfcc.shape[0] > MAX_MFCC_LEN:
                mfcc = mfcc[:MAX_MFCC_LEN, :]
            else:
                mfcc = np.pad(mfcc, ((0, MAX_MFCC_LEN - mfcc.shape[0]), (0, 0)), 'constant')
                
            # 5. Extract Statistical Features
            chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=SR).T, axis=0)
            contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=SR).T, axis=0)
            zcr = np.mean(librosa.feature.zero_crossing_rate(y=y).T, axis=0)
            rms = np.mean(librosa.feature.rms(y=y).T, axis=0)
            
            stats = np.hstack([chroma, contrast, zcr, rms])
            
            # 6. Append to lists 
            successful_rows.append(row)
            mfcc_data.append(mfcc.flatten())
            stats_data.append(stats)
            
        except Exception as e:
            continue

    # ==========================================
    # 3. BUILD ARRAYS & SAVE FOR THIS SPLIT
    # ==========================================
    if successful_rows:
        print(f"Constructing NPZ archive for {split.upper()}...")
        
        # Convert the original rows back into a DataFrame to format them cleanly
        df_meta = pd.DataFrame(successful_rows).reset_index(drop=True)
        if 'split' not in df_meta.columns:
            df_meta.insert(0, 'split', split)
            
        out_npz = f"dataset_meld/all_features_{split}_{timestamp}.npz"
        print(f"Saving {split.upper()} to {out_npz}...")

        # Convert numerical lists to strict numpy arrays (float32 saves 50% memory)
        np_mfcc = np.array(mfcc_data, dtype=np.float32)
        np_stats = np.array(stats_data, dtype=np.float32)
        
        # Convert metadata to numpy structures
        np_meta = df_meta.to_numpy(dtype=str)
        meta_cols = df_meta.columns.to_numpy(dtype=str)

        # Save as a compressed NumPy archive
        np.savez_compressed(
            out_npz,
            mfcc=np_mfcc,
            stats=np_stats,
            meta=np_meta,
            meta_cols=meta_cols
        )
        print(f"{split.upper()} Matrix shapes -> Meta: {np_meta.shape}, MFCC: {np_mfcc.shape}, Stats: {np_stats.shape}\n")
    else:
        print(f"No valid samples extracted for {split.upper()}.\n")

    # ==========================================
    # 4. MEMORY CLEANUP
    # ==========================================
    del successful_rows, mfcc_data, stats_data, df_meta, np_mfcc, np_stats, np_meta
    gc.collect()

print("All splits processed and saved independently as NPZ archives!")

# 3.0 Load and Import (features and functions)

### Import NPZs (labels + acoustic features + textual sentiment predicted)

In [ ]:
def load_and_prepare_npz(split, timestamp="20260713_2043"):
    path = os.path.join(Config.DATA_PATH, f"all_features_{split}_{timestamp}.npz")
    data = np.load(path, allow_pickle=True)
    
    # Reshape MFCCs back to 2D sequence (Batch, TimeSteps, Features)
    X_mfcc = data['mfcc'].reshape(-1, Config.MAX_MFCC_LEN, Config.N_MFCC)
    X_stats = data['stats']
    
    meta_cols = data['meta_cols'].tolist()
    
    # Extract Target Labels (Emotion)
    emo_idx = meta_cols.index('emotion')
    y_raw = data['meta'][:, emo_idx]
    
    # Extract Textual Sentiment Predicted strings (Bimodal setup)
    pred_idx = meta_cols.index('textual_sentiment_predicted')
    X_text_pred_raw = data['meta'][:, pred_idx]
    
    return X_mfcc, X_stats, X_text_pred_raw, y_raw

print("Loading Data...")
X_train_mfcc, X_train_stats, X_train_txt_raw, y_train_raw = load_and_prepare_npz('train')
X_dev_mfcc, X_dev_stats, X_dev_txt_raw, y_dev_raw = load_and_prepare_npz('dev')
X_test_mfcc, X_test_stats, X_test_txt_raw, y_test_raw = load_and_prepare_npz('test')

# --- ENCODE TARGETS (Emotion) ---
label_encoder = LabelEncoder()
label_encoder.fit(Config.DEFAULT_CLASSES)

y_train = label_encoder.transform(y_train_raw)
y_dev = label_encoder.transform(y_dev_raw)
y_test = label_encoder.transform(y_test_raw)
num_classes = len(Config.DEFAULT_CLASSES)

# --- ENCODE TEXT MODALITY (Sentiment Predictions) ---
# OneHotEncoder transforms 'positive', 'neutral', 'negative' into 2D numerical vectors 
sentiment_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# We reshape to (-1, 1) because OneHotEncoder expects a 2D array matrix 
X_train_txt = sentiment_encoder.fit_transform(X_train_txt_raw.reshape(-1, 1))
X_dev_txt = sentiment_encoder.transform(X_dev_txt_raw.reshape(-1, 1))
X_test_txt = sentiment_encoder.transform(X_test_txt_raw.reshape(-1, 1))

print(f"✅ Data Loaded & Encoded.")
print(f"Train MFCC Shape: {X_train_mfcc.shape}")
print(f"Train Stats Shape: {X_train_stats.shape}")
print(f"Train Text Modality Shape: {X_train_txt.shape} (One-Hot Encoded Sentiment)")

### Evaluation Functions

In [ ]:
def plot_training_history(history, model_name):
    """Plots training/validation accuracy and loss."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy Plot
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
    axes[0].set_title(f'{model_name} - Model Accuracy')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Loss Plot
    axes[1].plot(history.history['loss'], label='Train Loss', color='blue')
    axes[1].plot(history.history['val_loss'], label='Validation Loss', color='orange')
    axes[1].set_title(f'{model_name} - Model Loss')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

def evaluate_model_performance(model, X_test, y_test, model_name):
    """Generates Classification Report and Percentage-based Confusion Matrix."""
    print(f"\n================ Evaluating {model_name} ================")
    
    # Predictions
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # 1. Classification Report
    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    
    # 2. Confusion Matrix (Raw & Percentages)
    cm = confusion_matrix(y_test, y_pred)
    # Calculate percentages across rows (true labels)
    cm_percentages = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_percentages = np.nan_to_num(cm_percentages) # Handle division by zero
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_percentages, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Percentage of True Class'})
    plt.title(f'{model_name} - Confusion Matrix (Percentages)')
    plt.ylabel('Ground Truth (True Label)')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Section 3. Train and Test

## CNN-LSTM Model (Training & Testing)

In [ ]:
# TRAIN
cnn_lstm_model = build_hybrid_cnn_lstm(num_classes, X_train_stats.shape[1], X_train_txt.shape[1])

print(f"Training for exactly {Config.EPOCHS} epochs...")
history_cnn_lstm = cnn_lstm_model.fit(
    [X_train_mfcc, X_train_stats, X_train_txt], y_train,             
    validation_data=([X_dev_mfcc, X_dev_stats, X_dev_txt], y_dev),     
    epochs=Config.EPOCHS,
    batch_size=Config.BATCH_SIZE,
    verbose=1
)

In [ ]:
# ==========================================
# 4. EVALUATE
# ==========================================
plot_training_history(history_cnn_lstm, "Bimodal CNN-LSTM Model")
evaluate_model_performance(cnn_lstm_model, [X_test_mfcc, X_test_stats, X_test_txt], y_test, "Bimodal CNN-LSTM Model")

### 1. Modality ablation — test the "text branch is doing all the work" hypothesis
Run inference three times, each time zeroing out one branch's input, and compare accuracy drop:

In [ ]:
def eval_with_branch_masked(model, X_mfcc, X_stats, X_txt, y, mask_branch=None):
    Xm, Xs, Xt = X_mfcc.copy(), X_stats.copy(), X_txt.copy()
    if mask_branch == "mfcc":
        Xm[:] = 0
    elif mask_branch == "stats":
        Xs[:] = 0
    elif mask_branch == "text":
        Xt[:] = 0
    loss, acc = model.evaluate([Xm, Xs, Xt], y, verbose=0)
    return acc

baseline_acc = cnn_lstm_model.evaluate([X_dev_mfcc, X_dev_stats, X_dev_txt], y_dev, verbose=0)[1]
acc_no_mfcc  = eval_with_branch_masked(cnn_lstm_model, X_dev_mfcc, X_dev_stats, X_dev_txt, y_dev, "mfcc")
acc_no_stats = eval_with_branch_masked(cnn_lstm_model, X_dev_mfcc, X_dev_stats, X_dev_txt, y_dev, "stats")
acc_no_text  = eval_with_branch_masked(cnn_lstm_model, X_dev_mfcc, X_dev_stats, X_dev_txt, y_dev, "text")

print(f"Baseline (all branches):     {baseline_acc:.4f}")
print(f"Text branch zeroed:         {acc_no_text:.4f}  (drop = {baseline_acc-acc_no_text:.4f})")
print(f"MFCC branch zeroed:         {acc_no_mfcc:.4f}  (drop = {baseline_acc-acc_no_mfcc:.4f})")
print(f"Stats branch zeroed:        {acc_no_stats:.4f}  (drop = {baseline_acc-acc_no_stats:.4f})")

### 2. Permutation importance (more rigorous than zeroing)
Zeroing can be an out-of-distribution input the model has never seen. Permutation (shuffling that branch's values across the batch) keeps the same distribution:

In [ ]:
def permutation_importance(model, X_mfcc, X_stats, X_txt, y, branch, n_repeats=5):
    baseline = model.evaluate([X_mfcc, X_stats, X_txt], y, verbose=0)[1]
    drops = []
    for _ in range(n_repeats):
        idx = np.random.permutation(len(y))
        Xm, Xs, Xt = X_mfcc.copy(), X_stats.copy(), X_txt.copy()
        if branch == "mfcc": Xm = Xm[idx]
        elif branch == "stats": Xs = Xs[idx]
        elif branch == "text": Xt = Xt[idx]
        acc = model.evaluate([Xm, Xs, Xt], y, verbose=0)[1]
        drops.append(baseline - acc)
    return np.mean(drops), np.std(drops)

for branch in ["mfcc", "stats", "text"]:
    mean_drop, std_drop = permutation_importance(cnn_lstm_model, X_dev_mfcc, X_dev_stats, X_dev_txt, y_dev, branch)
    print(f"{branch:6s}: mean accuracy drop = {mean_drop:.4f} ± {std_drop:.4f}")

In [ ]:
print("Stats feature means:", X_dev_stats.mean(axis=0)[:5])
print("Stats feature stds:", X_dev_stats.std(axis=0)[:5])

### 3. Investigate why stats contributes nothing 
— worth checking whether the stats features themselves carry emotion-relevant signal at all, independent of the model:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
rf.fit(X_train_stats, y_train)
print("Stats-only RF dev accuracy:", rf.score(X_dev_stats, y_dev))

### 4. CONFUSION MATRIX: within-valence vs cross-valence error analysis

In [ ]:
# ==========================================
# CONFUSION MATRIX: within-valence vs cross-valence error analysis
# ==========================================
# Define emotion label order (must match your label encoding, e.g., LabelEncoder classes_)
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Get model predictions on the dev set
y_pred_probs = cnn_lstm_model.predict([X_dev_mfcc, X_dev_stats, X_dev_txt])
y_pred = np.argmax(y_pred_probs, axis=1)

# Full classification report
print("=" * 60)
print("CLASSIFICATION REPORT (Dev Set)")
print("=" * 60)
print(classification_report(y_dev, y_pred, target_names=emotion_labels, digits=3))

# Confusion matrix
cm = confusion_matrix(y_dev, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_labels, yticklabels=emotion_labels)
plt.title("Confusion Matrix — Dev Set")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

# ==========================================
# Group emotions into valence buckets and quantify within- vs cross-valence errors
# ==========================================
valence_map = {
    'joy': 'positive',
    'surprise': 'positive',      # surprise is ambiguous but often grouped positive in MELD's own sentiment mapping
    'neutral': 'neutral',
    'anger': 'negative',
    'disgust': 'negative',
    'fear': 'negative',
    'sadness': 'negative'
}

within_valence_errors = 0
cross_valence_errors = 0
correct = 0

for true_idx, pred_idx in zip(y_dev, y_pred):
    true_label = emotion_labels[true_idx]
    pred_label = emotion_labels[pred_idx]

    if true_label == pred_label:
        correct += 1
    elif valence_map[true_label] == valence_map[pred_label]:
        within_valence_errors += 1
    else:
        cross_valence_errors += 1

total = len(y_dev)
print("\n" + "=" * 60)
print("ERROR BREAKDOWN BY VALENCE")
print("=" * 60)
print(f"Correct predictions        : {correct} ({correct/total:.4f})")
print(f"Within-valence errors      : {within_valence_errors} ({within_valence_errors/total:.4f})")
print(f"Cross-valence errors       : {cross_valence_errors} ({cross_valence_errors/total:.4f})")
print(f"Total errors               : {within_valence_errors + cross_valence_errors}")
print(f"\nOf all errors, {within_valence_errors / (within_valence_errors + cross_valence_errors):.1%} stay within the same valence group")

## BAN Model (Training & Testing)

In [ ]:
# TRAIN
ban_model = build_ban(num_classes, X_train_stats.shape[1], X_train_txt.shape[1])

print(f"Training BAN for exactly {Config.EPOCHS} epochs...")
history_ban = ban_model.fit(
    [X_train_mfcc, X_train_stats, X_train_txt], y_train,  
    validation_data=([X_dev_mfcc, X_dev_stats, X_dev_txt], y_dev),
    epochs=Config.EPOCHS,
    batch_size=Config.BATCH_SIZE,
    verbose=1
)

In [ ]:
# ==========================================
# 5. EVALUATE
# ==========================================
plot_training_history(history_ban, "Bimodal Attention Network (BAN)")
evaluate_model_performance(ban_model, [X_test_mfcc, X_test_stats, X_test_txt], y_test, "Bimodal Attention Network (BAN)")